In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["groq_api_key"]=os.getenv("groq_api_key")

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

from langchain.chat_models import init_chat_model
model=init_chat_model("groq:openai/gpt-oss-20b")


In [4]:

def read_email_tool(email_id:str)->str:
	""" Mock function to read an email by its ID."""
	return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
	""" Mock function to send an email by its ID."""
	return f"Email sent to {recipient} with subject '{subject}'"

In [12]:
agent=create_agent(
model,
tools=[read_email_tool,send_email_tool],
checkpointer=InMemorySaver(),
middleware=[
HumanInTheLoopMiddleware(
interrupt_on={
"send_email_tool":{"allowed_decisions":["approved","edit","reject"]},
"read_email_tool":False,

})])

In [17]:
config={"configurable":{"thread_id":"test-approve"}}
##step1:Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="send email to john@twst.com with subject 'hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

##step2:Approve
from langgraph.types import Command

if "__interrupt__" in result:
	print("Puased approving...")
	result=agent.invoke(
		Command(resume={"decisions":[ {"type":"approve"}]},
		config=config))

print(f"Result:{result['messages'][-1].content}")

Result:Your email has been sent.
